In [0]:
# imports and installs 
%pip install azure-storage-blob
dbutils.library.restartPython()

In [0]:
# Configuration - All settings live here and can be modified in one place 
ACCOUNT_URL  = "https://zbskunkworks.blob.core.windows.net" # Storage account
CONTAINER    = "cost-management" # Top-level folder that holds exports
PREFIX       = "exports/" # Subfolder
VOLUME_ROOT  = "/Volumes/finops/raw/real_exports" # Destination - where exports land in Databricks

# Access to SAS token 
SECRET_SCOPE = "azure-cost" 
SECRET_KEY   = "skunkworks-sas"

In [0]:
# Establish connection to storage account 
from azure.storage.blob import ContainerClient

# Fetch SAS token 
# NEVER revealed in notebook or Git 
sas = dbutils.secrets.get(scope=SECRET_SCOPE, key=SECRET_KEY)

# Object used to list and download files 
container_client = ContainerClient(
    account_url=ACCOUNT_URL,   # which storage account
    container_name=CONTAINER,  # which container inside it
    credential=sas,            # key to access it 
)

# Test connection and SAS by listing all files under Exports (PREFIX)
for b in container_client.list_blobs(name_starts_with=PREFIX):
    print(b.size, b.name)

In [0]:
# Planning before Copying 

# For each source file, decide what the copy step should do:
#   new       -> not in the volume yet          -> COPY
#   changed   -> same name, different size      -> COPY (overwrite)
#   identical -> same name, same size           -> SKIP

# Limitation: Same name + same size + new content -> SKIP 

import os


def decide(blob):
    """Return 'new', 'changed' or 'identical' for one source blob."""
    # Mirror the source folder structure inside the volume, so
    # exports/x/y.csv.gz lands at /Volumes/.../exports/x/y.csv.gz
    local_path = f"{VOLUME_ROOT}/{blob.name}"

    # Volumes are mounted like a normal folder, so plain Python
    # file functions (os.path.*) work on /Volumes/... paths.
    if not os.path.exists(local_path):
        return "new - COPY"
    if os.path.getsize(local_path) != blob.size:
        return "changed - COPY (overwrite)"
    return "identical - SKIP"


# Collect one decision per source file into a list of dicts.
# The copy cell (cell 5) will read this list and act on it.
plan = []
for blob in container_client.list_blobs(name_starts_with=PREFIX):
    plan.append({
        "action": decide(blob),
        "name": blob.name,
        "size": blob.size,
    })

# Print the plan: action (padded for alignment), size, path.
for row in plan:
    print(row["action"].ljust(10), row["size"], row["name"])

# Summary count per action, a quick sanity check.
from collections import Counter
print("\nSummary:", dict(Counter(row["action"] for row in plan)))

In [0]:
# Copies external storage into my volume via SAS token
# Unity Catalog cannot read external storage, only access path is SAS token 

import os

copied, skipped = 0, 0

for row in plan:
    # Only copy new or changed files
    if row["action"] not in {"new", "changed"}:
        skipped += 1
        continue

    # Same folder layout as the source, inside the volume
    local_path = f"{VOLUME_ROOT}/{row['name']}"

    # Create the destination folders if they're missing
    os.makedirs(os.path.dirname(local_path), exist_ok=True)

    # Download the file and write it in binary (the CSVs are gzipped)
    with open(local_path, "wb") as f:
        container_client.download_blob(row["name"]).readinto(f)

    # Make sure the copy is the same size as the source
    if os.path.getsize(local_path) != row["size"]:
        raise RuntimeError(f"Size mismatch for {row['name']}")

    print("copied", row["size"], row["name"])
    copied += 1

print(f"Copied {copied}, skipped {skipped}")